In [43]:
import pandas as pd
# Read the given Data 
df = pd.read_csv('quick_hure.csv')

# Check for missing Values
print('number of misssing values present in the dataset are \n' , df.isnull().sum()) 

#check what are the diffrent categories available
print('number of categories present',df['Category'].unique())

# print the data schema 
df.head()




number of misssing values present in the dataset are 
 State       0
Date        0
Total       0
Category    0
dtype: int64
number of categories present <StringArray>
['Beverages']
Length: 1, dtype: str


,State,Date,Total,Category
0,Alabama,1/12/2019,"109,574,036",Beverages
1,Arizona,1/12/2019,"109,101,595",Beverages
2,Arkansas,1/12/2019,"58,049,432",Beverages
3,California,1/12/2019,"444,766,891",Beverages
4,Colorado,1/12/2019,"89,816,716",Beverages


In [44]:
# cleaning the dataset 

#dropping un-impactful collumns to reduce dimensions
df.drop(columns=['Category'] , inplace = True )
df.head()


,State,Date,Total
0,Alabama,1/12/2019,"109,574,036"
1,Arizona,1/12/2019,"109,101,595"
2,Arkansas,1/12/2019,"58,049,432"
3,California,1/12/2019,"444,766,891"
4,Colorado,1/12/2019,"89,816,716"


In [45]:
df['Date'] = pd.to_datetime(df['Date'] , format='mixed' , dayfirst=True) 
df.head()
print('dtype of date object: ', df['Date'].dtype)


#sorting the values in ascending order 
df = df.sort_values(['State' , 'Date'])
print(df.head())


dtype of date object:  datetime64[us]
        State       Date           Total
86    Alabama 2019-10-06    129,106,730 
3397  Alabama 2019-10-13    123,782,286 
5246  Alabama 2019-10-20    116,218,909 
7095  Alabama 2019-10-27    109,968,011 
43    Alabama 2019-11-03    112,189,104 


In [46]:
# check if there are any missing dates

for state, group in df.groupby('State'):

    expected = pd.date_range(
        start=group['Date'].min(),
        end=group['Date'].max(),
        freq='W'
    )

    missing = expected.difference(group['Date'])

    if len(missing) > 0:
        print(state)
        print(missing)

In [47]:
# Creating the lag features
df['lag_1'] = df['Total'].shift(1)
df['lag_7'] = df['Total'].shift(7)
df['lag_30'] = df['Total'].shift(30)
df['month'] = df['Date'].dt.month
df['week'] = df['Date'].dt.isocalendar().week
df['quarter'] = df['Date'].dt.quarter
df['year'] = df['Date'].dt.year
df.head()


,State,Date,Total,lag_1,lag_7,lag_30,month,week,quarter,year
86,Alabama,2019-10-06,"129,106,730",NaN,NaN,NaN,10,40,4,2019
3397,Alabama,2019-10-13,"123,782,286","129,106,730",NaN,NaN,10,41,4,2019
5246,Alabama,2019-10-20,"116,218,909","123,782,286",NaN,NaN,10,42,4,2019
7095,Alabama,2019-10-27,"109,968,011","116,218,909",NaN,NaN,10,43,4,2019
43,Alabama,2019-11-03,"112,189,104","109,968,011",NaN,NaN,11,44,4,2019


In [48]:
# adding holiday features for the US 
import holidays

us_holidays = holidays.US()

df['is_holiday'] = (
    df['Date'].isin(us_holidays)
).astype(int)

In [49]:
df['Total'] = df['Total'].str.replace(',', '').astype(float)

df['lag_1'] = df['lag_1'].str.replace(',', '').astype(float)

df['lag_7'] = df['lag_7'].str.replace(',', '').astype(float)

df['lag_30'] = df['lag_30'].str.replace(',', '').astype(float)
df.head()


,State,Date,Total,lag_1,lag_7,lag_30,month,week,quarter,year,is_holiday
86,Alabama,2019-10-06,129106730.0,NaN,NaN,NaN,10,40,4,2019,0
3397,Alabama,2019-10-13,123782286.0,129106730.0,NaN,NaN,10,41,4,2019,0
5246,Alabama,2019-10-20,116218909.0,123782286.0,NaN,NaN,10,42,4,2019,0
7095,Alabama,2019-10-27,109968011.0,116218909.0,NaN,NaN,10,43,4,2019,0
43,Alabama,2019-11-03,112189104.0,109968011.0,NaN,NaN,11,44,4,2019,0


In [50]:
df.to_csv('cleaned_data.csv')

In [51]:
# splitting the data 
df.set_index('Date' , inplace= True)
train_size = int(len(df) * 0.8)

train = df.iloc[:train_size]

test = df.iloc[train_size:]

In [52]:
print(train.head())

              State        Total        lag_1  lag_7  lag_30  month  week  \
Date                                                                        
2019-10-06  Alabama  129106730.0          NaN    NaN     NaN     10    40   
2019-10-13  Alabama  123782286.0  129106730.0    NaN     NaN     10    41   
2019-10-20  Alabama  116218909.0  123782286.0    NaN     NaN     10    42   
2019-10-27  Alabama  109968011.0  116218909.0    NaN     NaN     10    43   
2019-11-03  Alabama  112189104.0  109968011.0    NaN     NaN     11    44   

            quarter  year  is_holiday  
Date                                   
2019-10-06        4  2019           0  
2019-10-13        4  2019           0  
2019-10-20        4  2019           0  
2019-10-27        4  2019           0  
2019-11-03        4  2019           0  


In [53]:
# =========================
# Faster SARIMAX Training
# =========================

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler

# -------------------------
# Features
# -------------------------

exog_cols = [
    'lag_1',
    'lag_7',
    'month',
    'week',
    'is_holiday'
]

# -------------------------
# Drop missing rows
# -------------------------

train_model = train.dropna(subset=exog_cols)
test_model = test.dropna(subset=exog_cols)

# -------------------------
# Target
# -------------------------

y_train = train_model['Total'].astype(float)
y_test = test_model['Total'].astype(float)

# -------------------------
# Exogenous variables
# -------------------------

X_train = train_model[exog_cols].astype(float)
X_test = test_model[exog_cols].astype(float)

# -------------------------
# Scale target
# -------------------------

scaler = StandardScaler()

y_train_scaled = scaler.fit_transform(
    y_train.values.reshape(-1,1)
).flatten()

# -------------------------
# Build Faster Model
# -------------------------

model = SARIMAX(
    y_train_scaled,
    exog=X_train,
    order=(1,1,1),

    # removed heavy yearly seasonality
    seasonal_order=(0,0,0,0),

    enforce_stationarity=False,
    enforce_invertibility=False
)

# -------------------------
# Train
# -------------------------

results = model.fit(
    maxiter=30,
    disp=False
)

# -------------------------
# Forecast
# -------------------------

pred_scaled = results.forecast(
    steps=len(X_test),
    exog=X_test
)

# -------------------------
# Inverse scaling
# -------------------------

predictions = scaler.inverse_transform(
    pred_scaled.values.reshape(-1,1)
).flatten()

# -------------------------
# Evaluate
# -------------------------

mae = mean_absolute_error(
    y_test,
    predictions
)

print("MAE:", mae)

e:\Gowtham\torch-env\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
e:\Gowtham\torch-env\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it is not monotonic and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
e:\Gowtham\torch-env\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
e:\Gowtham\torch-env\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it is not monotonic and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
e:\Gowtham\torch-env\Lib\site-packages\statsmodels

MAE: 138320953.0078529


e:\Gowtham\torch-env\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
e:\Gowtham\torch-env\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [54]:
# %pip install csv

def save_mod_per(model , val):
  df = pd.DataFrame({'Model': [model], 'Value': [val]})
  df.to_csv('model_per.csv' , mode ='a', index=False , header=False )
   


In [55]:
print(mae)
save_mod_per('SARIMA' , mae)

138320953.0078529


In [56]:
#prophet
